In [1]:
import sys

sys.path.append("..")

from common.metrics import metrics
from common.preprocessing import preprocessing_cn7
from common.train_test_split import split_data
from common.model_composition import model_composition

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_cn7 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv")

In [3]:
X_cn7, y_cn7 = preprocessing_cn7(train_cn7)

print("X shape:", X_cn7.shape)
print("y shape:", y_cn7.shape)

X shape: (1211, 23)
y shape: (1211,)


Hard defect = Train 내부 OOF Logistic에서 불량확률이 낮은 하위 50% 불량으로 정의

In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier


# ============================================================
# 0. 데이터
# ============================================================

X_array = X_cn7.to_numpy(dtype=np.float64)
y_array = y_cn7.to_numpy()


# ============================================================
# 1. Outer CV
# ============================================================

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


models = {
    "XGBoost": lambda: XGBClassifier(
        random_state=42
    ),

    "CatBoost": lambda: CatBoostClassifier(
        random_state=42,
        verbose=0
    )
}


results = []


# ============================================================
# 2. 모델별 실험
# ============================================================

for model_name, model_fn in models.items():

    baseline_prob = np.zeros(len(y_array))
    hard_ros_prob = np.zeros(len(y_array))


    for outer_fold, (
        train_idx,
        val_idx
    ) in enumerate(
        outer_cv.split(
            X_array,
            y_array
        ),
        start=1
    ):

        X_train = X_array[train_idx]
        y_train = y_array[train_idx]

        X_val = X_array[val_idx]


        # ====================================================
        # 3. Baseline
        # ====================================================

        baseline_model = model_fn()

        baseline_model.fit(
            X_train,
            y_train
        )

        baseline_prob[
            val_idx
        ] = baseline_model.predict_proba(
            X_val
        )[:, 1]


        # ====================================================
        # 4. Train 내부에서 Hard defect 찾기
        # ====================================================

        inner_cv = StratifiedKFold(
            n_splits=4,
            shuffle=True,
            random_state=100 + outer_fold
        )

        train_oof_prob = np.zeros(
            len(y_train)
        )


        for (
            inner_train_idx,
            inner_val_idx
        ) in inner_cv.split(
            X_train,
            y_train
        ):

            hardness_model = LogisticRegression(
                max_iter=1000,
                random_state=42
            )

            hardness_model.fit(
                X_train[inner_train_idx],
                y_train[inner_train_idx]
            )

            train_oof_prob[
                inner_val_idx
            ] = hardness_model.predict_proba(
                X_train[inner_val_idx]
            )[:, 1]


        # ====================================================
        # 5. 불량 중 확률이 낮은 하위 50% = Hard defect
        # ====================================================

        defect_local_idx = np.where(
            y_train == 1
        )[0]

        defect_prob = train_oof_prob[
            defect_local_idx
        ]


        hard_cutoff = np.median(
            defect_prob
        )


        hard_local_idx = defect_local_idx[
            defect_prob <= hard_cutoff
        ]


        print(
            f"{model_name} | Fold {outer_fold} | "
            f"Defects: {len(defect_local_idx)} | "
            f"Hard defects: {len(hard_local_idx)} | "
            f"Cutoff: {hard_cutoff:.6f}"
        )


        # ====================================================
        # 6. Hard defect만 선택적으로 복제
        #
        # 원본 + Hard defect 5회 추가
        # ====================================================

        repeat_count = 5

        X_hard = X_train[
            hard_local_idx
        ]

        y_hard = y_train[
            hard_local_idx
        ]


        X_aug = np.concatenate(
            [
                X_train,
                np.repeat(
                    X_hard,
                    repeat_count,
                    axis=0
                )
            ],
            axis=0
        )

        y_aug = np.concatenate(
            [
                y_train,
                np.repeat(
                    y_hard,
                    repeat_count
                )
            ]
        )


        # ====================================================
        # 7. Hard ROS 모델 학습
        # ====================================================

        hard_model = model_fn()

        hard_model.fit(
            X_aug,
            y_aug
        )


        hard_ros_prob[
            val_idx
        ] = hard_model.predict_proba(
            X_val
        )[:, 1]


    # ========================================================
    # 8. Threshold 0.5 성능 함수
    # ========================================================

    def evaluate(
        y_true,
        prob
    ):

        pred = (
            prob >= 0.5
        ).astype(int)

        return {
            "PR-AUC":
                average_precision_score(
                    y_true,
                    prob
                ),

            "Precision":
                precision_score(
                    y_true,
                    pred,
                    zero_division=0
                ),

            "Recall":
                recall_score(
                    y_true,
                    pred,
                    zero_division=0
                ),

            "F1":
                f1_score(
                    y_true,
                    pred,
                    zero_division=0
                )
        }


    baseline_result = evaluate(
        y_array,
        baseline_prob
    )

    hard_result = evaluate(
        y_array,
        hard_ros_prob
    )


    results.append({
        "Model": model_name,
        "Method": "Baseline",
        **baseline_result
    })

    results.append({
        "Model": model_name,
        "Method": "Hard-defect ROS",
        **hard_result
    })


# ============================================================
# 9. 비교
# ============================================================

hard_ros_result_df = pd.DataFrame(
    results
)

print(
    "\n[Hard-defect ROS Result]"
)

display(
    hard_ros_result_df
)

XGBoost | Fold 1 | Defects: 13 | Hard defects: 7 | Cutoff: 0.168339
XGBoost | Fold 2 | Defects: 14 | Hard defects: 7 | Cutoff: 0.112500
XGBoost | Fold 3 | Defects: 14 | Hard defects: 7 | Cutoff: 0.043836
XGBoost | Fold 4 | Defects: 14 | Hard defects: 7 | Cutoff: 0.087655
XGBoost | Fold 5 | Defects: 13 | Hard defects: 7 | Cutoff: 0.029733
CatBoost | Fold 1 | Defects: 13 | Hard defects: 7 | Cutoff: 0.168339
CatBoost | Fold 2 | Defects: 14 | Hard defects: 7 | Cutoff: 0.112500
CatBoost | Fold 3 | Defects: 14 | Hard defects: 7 | Cutoff: 0.043836
CatBoost | Fold 4 | Defects: 14 | Hard defects: 7 | Cutoff: 0.087655
CatBoost | Fold 5 | Defects: 13 | Hard defects: 7 | Cutoff: 0.029733

[Hard-defect ROS Result]


,Model,Method,PR-AUC,Precision,Recall,F1
0,XGBoost,Baseline,0.406236,0.375,0.352941,0.363636
1,XGBoost,Hard-defect ROS,0.204577,0.375,0.352941,0.363636
2,CatBoost,Baseline,0.440622,0.375,0.352941,0.363636
3,CatBoost,Hard-defect ROS,0.232000,0.375,0.352941,0.363636


Hard-defect Jitter

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier


# ============================================================
# 0. 데이터
# ============================================================

X_array = X_cn7.to_numpy(dtype=np.float64)
y_array = y_cn7.to_numpy()


# ============================================================
# 1. Outer CV
# ============================================================

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


models = {
    "XGBoost": lambda: XGBClassifier(
        random_state=42
    ),

    "CatBoost": lambda: CatBoostClassifier(
        random_state=42,
        verbose=0
    )
}


results = []


# ============================================================
# 2. 모델별 실험
# ============================================================

for model_name, model_fn in models.items():

    baseline_prob = np.zeros(len(y_array))
    jitter_prob = np.zeros(len(y_array))


    for outer_fold, (train_idx, val_idx) in enumerate(
        outer_cv.split(X_array, y_array),
        start=1
    ):

        X_train = X_array[train_idx]
        y_train = y_array[train_idx]

        X_val = X_array[val_idx]


        # ====================================================
        # 3. Baseline
        # ====================================================

        baseline_model = model_fn()

        baseline_model.fit(
            X_train,
            y_train
        )

        baseline_prob[val_idx] = baseline_model.predict_proba(
            X_val
        )[:, 1]


        # ====================================================
        # 4. Train 내부 OOF로 Hard defect 선정
        # ====================================================

        inner_cv = StratifiedKFold(
            n_splits=4,
            shuffle=True,
            random_state=100 + outer_fold
        )

        train_oof_prob = np.zeros(
            len(y_train)
        )


        for inner_train_idx, inner_val_idx in inner_cv.split(
            X_train,
            y_train
        ):

            hardness_model = LogisticRegression(
                max_iter=1000,
                random_state=42
            )

            hardness_model.fit(
                X_train[inner_train_idx],
                y_train[inner_train_idx]
            )

            train_oof_prob[inner_val_idx] = (
                hardness_model.predict_proba(
                    X_train[inner_val_idx]
                )[:, 1]
            )


        # ====================================================
        # 5. 불량 중 확률 하위 50% = Hard defect
        # ====================================================

        defect_local_idx = np.where(
            y_train == 1
        )[0]

        defect_prob = train_oof_prob[
            defect_local_idx
        ]

        hard_cutoff = np.median(
            defect_prob
        )

        hard_local_idx = defect_local_idx[
            defect_prob <= hard_cutoff
        ]


        print(
            f"{model_name} | Fold {outer_fold} | "
            f"Defects: {len(defect_local_idx)} | "
            f"Hard defects: {len(hard_local_idx)} | "
            f"Cutoff: {hard_cutoff:.6f}"
        )


        # ====================================================
        # 6. Hard-defect Jitter 생성
        # ====================================================

        X_hard = X_train[
            hard_local_idx
        ]


        # Hard defect 하나당 합성 데이터 2개
        n_synthetic = 2

        # Standard Scaling 데이터 기준 작은 noise
        noise_std = 0.05

        rng = np.random.default_rng(
            42 + outer_fold
        )


        synthetic_list = []


        for x in X_hard:

            for _ in range(n_synthetic):

                noise = rng.normal(
                    loc=0.0,
                    scale=noise_std,
                    size=x.shape
                )

                synthetic_x = x + noise

                synthetic_list.append(
                    synthetic_x
                )


        X_synthetic = np.array(
            synthetic_list
        )

        y_synthetic = np.ones(
            len(X_synthetic),
            dtype=y_train.dtype
        )


        # ====================================================
        # 7. 원본 Train + 합성 Hard defect
        # ====================================================

        X_aug = np.concatenate(
            [
                X_train,
                X_synthetic
            ],
            axis=0
        )

        y_aug = np.concatenate(
            [
                y_train,
                y_synthetic
            ]
        )


        print(
            f"  Synthetic: {len(X_synthetic)} | "
            f"Train: {len(X_train)} -> {len(X_aug)}"
        )


        # ====================================================
        # 8. Jitter 모델 학습
        # ====================================================

        jitter_model = model_fn()

        jitter_model.fit(
            X_aug,
            y_aug
        )


        jitter_prob[val_idx] = jitter_model.predict_proba(
            X_val
        )[:, 1]


    # ========================================================
    # 9. 평가
    # ========================================================

    def evaluate(y_true, prob):

        pred = (
            prob >= 0.5
        ).astype(int)

        return {
            "PR-AUC": average_precision_score(
                y_true,
                prob
            ),

            "Precision": precision_score(
                y_true,
                pred,
                zero_division=0
            ),

            "Recall": recall_score(
                y_true,
                pred,
                zero_division=0
            ),

            "F1": f1_score(
                y_true,
                pred,
                zero_division=0
            )
        }


    baseline_result = evaluate(
        y_array,
        baseline_prob
    )

    jitter_result = evaluate(
        y_array,
        jitter_prob
    )


    results.append({
        "Model": model_name,
        "Method": "Baseline",
        **baseline_result
    })

    results.append({
        "Model": model_name,
        "Method": "Hard-defect Jitter",
        **jitter_result
    })


# ============================================================
# 10. 결과
# ============================================================

hard_jitter_result_df = pd.DataFrame(
    results
)

print(
    "\n[Hard-defect Jitter Result]"
)

display(
    hard_jitter_result_df
33)

XGBoost | Fold 1 | Defects: 13 | Hard defects: 7 | Cutoff: 0.168339
  Synthetic: 14 | Train: 968 -> 982
XGBoost | Fold 2 | Defects: 14 | Hard defects: 7 | Cutoff: 0.112500
  Synthetic: 14 | Train: 969 -> 983
XGBoost | Fold 3 | Defects: 14 | Hard defects: 7 | Cutoff: 0.043836
  Synthetic: 14 | Train: 969 -> 983
XGBoost | Fold 4 | Defects: 14 | Hard defects: 7 | Cutoff: 0.087655
  Synthetic: 14 | Train: 969 -> 983
XGBoost | Fold 5 | Defects: 13 | Hard defects: 7 | Cutoff: 0.029733
  Synthetic: 14 | Train: 969 -> 983
CatBoost | Fold 1 | Defects: 13 | Hard defects: 7 | Cutoff: 0.168339
  Synthetic: 14 | Train: 968 -> 982
CatBoost | Fold 2 | Defects: 14 | Hard defects: 7 | Cutoff: 0.112500
  Synthetic: 14 | Train: 969 -> 983
CatBoost | Fold 3 | Defects: 14 | Hard defects: 7 | Cutoff: 0.043836
  Synthetic: 14 | Train: 969 -> 983
CatBoost | Fold 4 | Defects: 14 | Hard defects: 7 | Cutoff: 0.087655
  Synthetic: 14 | Train: 969 -> 983
CatBoost | Fold 5 | Defects: 13 | Hard defects: 7 | Cutoff: 

,Model,Method,PR-AUC,Precision,Recall,F1
0,XGBoost,Baseline,0.406236,0.375,0.352941,0.363636
1,XGBoost,Hard-defect Jitter,0.313658,0.375,0.352941,0.363636
2,CatBoost,Baseline,0.440622,0.375,0.352941,0.363636
3,CatBoost,Hard-defect Jitter,0.432842,0.375,0.352941,0.363636


### CatBoost가 Logistic과 실제로 다른 불량을 잡는지부터 확인

In [6]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score


# ============================================================
# 0. 데이터
# ============================================================

X_array = X_cn7.to_numpy(dtype=np.float64)
y_array = y_cn7.to_numpy()


# ============================================================
# 1. CV
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_logistic = np.zeros(len(y_array))
oof_catboost = np.zeros(len(y_array))


# ============================================================
# 2. OOF 예측
# ============================================================

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_array, y_array),
    start=1
):

    print(f"Fold {fold}")

    X_train = X_array[train_idx]
    X_val = X_array[val_idx]

    y_train = y_array[train_idx]


    # Logistic
    logistic = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    logistic.fit(
        X_train,
        y_train
    )

    oof_logistic[val_idx] = logistic.predict_proba(
        X_val
    )[:, 1]


    # CatBoost
    catboost = CatBoostClassifier(
        random_state=42,
        verbose=0
    )

    catboost.fit(
        X_train,
        y_train
    )

    oof_catboost[val_idx] = catboost.predict_proba(
        X_val
    )[:, 1]


# ============================================================
# 3. PR-AUC
# ============================================================

print(
    "\nLogistic PR-AUC:",
    average_precision_score(
        y_array,
        oof_logistic
    )
)

print(
    "CatBoost PR-AUC:",
    average_precision_score(
        y_array,
        oof_catboost
    )
)


# ============================================================
# 4. Rank 계산
# ============================================================

compare_df = pd.DataFrame({
    "Target": y_array,

    "Logistic_Prob":
        oof_logistic,

    "CatBoost_Prob":
        oof_catboost
})


compare_df["Logistic_Rank"] = (
    compare_df["Logistic_Prob"]
    .rank(
        ascending=False,
        method="min"
    )
)

compare_df["CatBoost_Rank"] = (
    compare_df["CatBoost_Prob"]
    .rank(
        ascending=False,
        method="min"
    )
)


# CatBoost가 Logistic보다 얼마나 순위를 올렸는지
compare_df["Rank_Improvement"] = (
    compare_df["Logistic_Rank"]
    - compare_df["CatBoost_Rank"]
)


# ============================================================
# 5. 불량 17개만 확인
# ============================================================

defect_compare = (
    compare_df[
        compare_df["Target"] == 1
    ]
    .sort_values(
        "CatBoost_Rank"
    )
)


print(
    "\n[Logistic vs CatBoost - Defect Rank]"
)

display(
    defect_compare
)

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

Logistic PR-AUC: 0.46941459686939674
CatBoost PR-AUC: 0.4406215671635587

[Logistic vs CatBoost - Defect Rank]


,Target,Logistic_Prob,CatBoost_Prob,Logistic_Rank,CatBoost_Rank,Rank_Improvement
117,1,0.901290,0.980738,6.0,1.0,5.0
118,1,0.957725,0.978171,5.0,2.0,3.0
120,1,0.959126,0.973211,3.0,3.0,0.0
119,1,0.958082,0.972426,4.0,4.0,0.0
116,1,0.984080,0.965167,1.0,5.0,-4.0
115,1,0.963258,0.959822,2.0,6.0,-4.0
95,1,0.080410,0.014317,37.0,45.0,-8.0
113,1,0.031087,0.011548,92.0,56.0,36.0
99,1,0.021629,0.011544,119.0,57.0,62.0
103,1,0.079896,0.011011,38.0,61.0,-23.0


In [ ]:
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score
import pandas as pd


# ============================================================
# 1. 확률 → Percentile Rank
# ============================================================

logistic_rank_score = rankdata(
    oof_logistic,
    method="average"
) / len(oof_logistic)

catboost_rank_score = rankdata(
    oof_catboost,
    method="average"
) / len(oof_catboost)


# ============================================================
# 2. OR Rank Ensemble
# 둘 중 하나라도 높은 위험도를 주면 보존
# ============================================================

or_rank_score = np.maximum(
    logistic_rank_score,
    catboost_rank_score
)


# ============================================================
# 3. PR-AUC
# ============================================================

print(
    "Logistic PR-AUC:",
    average_precision_score(
        y_array,
        oof_logistic
    )
)

print(
    "CatBoost PR-AUC:",
    average_precision_score(
        y_array,
        oof_catboost
    )
)

print(
    "OR Rank Ensemble PR-AUC:",
    average_precision_score(
        y_array,
        or_rank_score
    )
)


# ============================================================
# 4. Ensemble Rank
# ============================================================

ensemble_df = pd.DataFrame({
    "Target": y_array,
    "Logistic_Rank": compare_df["Logistic_Rank"],
    "CatBoost_Rank": compare_df["CatBoost_Rank"],
    "OR_Score": or_rank_score
})


ensemble_df["OR_Rank"] = (
    ensemble_df["OR_Score"]
    .rank(
        ascending=False,
        method="min"
    )
)


# ============================================================
# 5. 불량만 출력
# ============================================================

defect_ensemble_df = (
    ensemble_df[
        ensemble_df["Target"] == 1
    ]
    .sort_values("OR_Rank")
)


print(
    "\n[Logistic + CatBoost OR Rank]"
)

display(
    defect_ensemble_df
)

Logistic PR-AUC: 0.46941459686939674
CatBoost PR-AUC: 0.4406215671635587
OR Rank Ensemble PR-AUC: 0.45470766491830933

[Logistic + CatBoost OR Rank]


,Target,Logistic_Rank,CatBoost_Rank,OR_Score,OR_Rank
116,1,1.0,5.0,1.000000,1.0
117,1,6.0,1.0,1.000000,1.0
118,1,5.0,2.0,0.999174,3.0
115,1,2.0,6.0,0.999174,3.0
120,1,3.0,3.0,0.998348,5.0
119,1,4.0,4.0,0.997523,6.0
109,1,23.0,133.0,0.981833,31.0
47,1,24.0,145.0,0.981007,33.0
95,1,37.0,45.0,0.970273,52.0
103,1,38.0,61.0,0.969447,53.0


### Nearest Normal Distance + 가장 가까운 정상의 index + 거리 순위

In [8]:
import numpy as np
import pandas as pd

from sklearn.neighbors import NearestNeighbors


# ============================================================
# 0. 데이터
# ============================================================

X_array = X_cn7.to_numpy(dtype=np.float64)
y_array = y_cn7.to_numpy()

original_index = np.array(X_cn7.index)


# ============================================================
# 1. 정상 / 불량 분리
# ============================================================

normal_mask = (y_array == 0)
defect_mask = (y_array == 1)

X_normal = X_array[normal_mask]
X_defect = X_array[defect_mask]

normal_index = original_index[normal_mask]
defect_index = original_index[defect_mask]


print("Normal :", len(X_normal))
print("Defect :", len(X_defect))


# ============================================================
# 2. 각 불량에서 가장 가까운 정상 5개 탐색
# ============================================================

nn = NearestNeighbors(
    n_neighbors=5,
    metric="euclidean"
)

nn.fit(X_normal)

distances, neighbor_idx = nn.kneighbors(
    X_defect
)


# ============================================================
# 3. 결과 DataFrame
# ============================================================

distance_df = pd.DataFrame({
    "Defect_Index": defect_index,

    # 가장 가까운 정상까지 거리
    "Nearest_Normal_Distance": distances[:, 0],

    # 가장 가까운 정상 index
    "Nearest_Normal_Index": normal_index[
        neighbor_idx[:, 0]
    ],

    # 가까운 정상 5개의 평균 거리
    "Mean_5NN_Distance": distances.mean(axis=1),

    # 두 번째로 가까운 정상
    "Second_Normal_Distance": distances[:, 1],

    # 다섯 번째로 가까운 정상
    "Fifth_Normal_Distance": distances[:, 4]
})


# ============================================================
# 4. 정상에서 멀리 떨어진 불량부터 정렬
# ============================================================

distance_df = distance_df.sort_values(
    "Nearest_Normal_Distance",
    ascending=False
).reset_index(drop=True)


distance_df["Distance_Rank"] = (
    distance_df["Nearest_Normal_Distance"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)


print(
    "\n[Defect → Nearest Normal Distance]"
)

display(distance_df)

Normal : 1194
Defect : 17

[Defect → Nearest Normal Distance]


,Defect_Index,Nearest_Normal_Distance,Nearest_Normal_Index,Mean_5NN_Distance,Second_Normal_Distance,Fifth_Normal_Distance,Distance_Rank
0,117,1.184889e+01,921,11.912536,11.848889,11.998360,1
1,118,1.184889e+01,921,11.912536,11.848889,11.998360,1
2,119,1.175281e+01,921,11.814305,11.752813,11.884909,3
3,120,1.175281e+01,921,11.814305,11.752813,11.884909,3
4,116,1.128755e+01,1155,11.343437,11.287553,11.392057,5
5,115,1.128755e+01,1155,11.343437,11.287553,11.392057,5
6,111,8.429370e-08,112,2.189006,2.351400,3.121115,7
7,107,8.429370e-08,108,2.517191,2.727592,3.565385,7
8,103,8.429370e-08,104,1.984309,2.462789,2.497984,7
9,47,0.000000e+00,48,1.936311,2.389373,2.451403,10


In [9]:
# ============================================================
# 정상 데이터끼리의 최근접 거리
#
# 자기 자신이 1번째 이웃이므로
# n_neighbors=2 → 두 번째 이웃 사용
# ============================================================

normal_nn = NearestNeighbors(
    n_neighbors=2,
    metric="euclidean"
)

normal_nn.fit(X_normal)

normal_distances, _ = normal_nn.kneighbors(
    X_normal
)

normal_nearest_distance = normal_distances[:, 1]


# ============================================================
# 정상 거리 분포
# ============================================================

normal_stats = pd.Series(
    normal_nearest_distance
).describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

print(
    "\n[Normal → Nearest Normal Distance]"
)

display(normal_stats)


# ============================================================
# 각 불량이 정상 거리분포의 몇 percentile인지 계산
# ============================================================

distance_df["Normal_Distance_Percentile"] = (
    distance_df["Nearest_Normal_Distance"]
    .apply(
        lambda x:
        np.mean(
            normal_nearest_distance <= x
        ) * 100
    )
)


print(
    "\n[Defect Distance Percentile]"
)

display(
    distance_df[
        [
            "Defect_Index",
            "Nearest_Normal_Distance",
            "Mean_5NN_Distance",
            "Nearest_Normal_Index",
            "Normal_Distance_Percentile",
            "Distance_Rank"
        ]
    ]
)


[Normal → Nearest Normal Distance]


count    1.194000e+03
mean     2.778083e-02
std      2.810369e-01
min      0.000000e+00
50%      0.000000e+00
75%      5.960464e-08
90%      8.429370e-08
95%      9.092430e-08
99%      1.504130e-01
max      4.287504e+00
dtype: float64


[Defect Distance Percentile]


,Defect_Index,Nearest_Normal_Distance,Mean_5NN_Distance,Nearest_Normal_Index,Normal_Distance_Percentile,Distance_Rank
0,117,1.184889e+01,11.912536,921,100.000000,1
1,118,1.184889e+01,11.912536,921,100.000000,1
2,119,1.175281e+01,11.814305,921,100.000000,3
3,120,1.175281e+01,11.814305,921,100.000000,3
4,116,1.128755e+01,11.343437,1155,100.000000,5
5,115,1.128755e+01,11.343437,1155,100.000000,5
6,111,8.429370e-08,2.189006,112,94.974874,7
7,107,8.429370e-08,2.517191,108,94.974874,7
8,103,8.429370e-08,1.984309,104,94.974874,7
9,47,0.000000e+00,1.936311,48,70.854271,10


### exact equality를 검증

In [10]:
# ============================================================
# 불량별 최근접 정상과 실제 컬럼 값이 동일한지 확인
# ============================================================

check_rows = []

for _, row in distance_df.iterrows():

    defect_idx = int(row["Defect_Index"])
    normal_idx = int(row["Nearest_Normal_Index"])

    defect_x = X_cn7.loc[defect_idx]
    normal_x = X_cn7.loc[normal_idx]

    diff = (defect_x - normal_x).abs()

    check_rows.append({
        "Defect_Index": defect_idx,
        "Normal_Index": normal_idx,
        "Distance": row["Nearest_Normal_Distance"],
        "Exact_Equal": defect_x.equals(normal_x),
        "Max_Abs_Diff": diff.max(),
        "Different_Columns": (diff > 1e-10).sum()
    })


exact_check_df = pd.DataFrame(check_rows)

display(
    exact_check_df.sort_values(
        "Distance",
        ascending=False
    )
)

,Defect_Index,Normal_Index,Distance,Exact_Equal,Max_Abs_Diff,Different_Columns
0,117,921,1.184889e+01,False,8.761174,21
1,118,921,1.184889e+01,False,8.761174,21
2,119,921,1.175281e+01,False,8.566476,20
3,120,921,1.175281e+01,False,8.566476,20
4,116,1155,1.128755e+01,False,8.371787,18
5,115,1155,1.128755e+01,False,8.371787,18
6,111,112,8.429370e-08,True,0.000000,0
7,107,108,8.429370e-08,True,0.000000,0
8,103,104,8.429370e-08,True,0.000000,0
9,47,48,0.000000e+00,True,0.000000,0


In [11]:
# ============================================================
# 동일 X인데 Target이 다른 그룹 확인
# ============================================================

temp = X_cn7.copy()
temp["PassOrFail"] = y_cn7.values


conflict_summary = (
    temp
    .groupby(
        list(X_cn7.columns),
        dropna=False
    )["PassOrFail"]
    .agg(
        Count="size",
        Target_Nunique="nunique",
        Normal_Count=lambda x: (x == 0).sum(),
        Defect_Count=lambda x: (x == 1).sum()
    )
    .reset_index()
)


conflict_groups = conflict_summary[
    conflict_summary["Target_Nunique"] > 1
]


print("충돌 X 그룹 수 :", len(conflict_groups))
print("충돌 그룹 전체 행 수 :", conflict_groups["Count"].sum())
print("충돌 그룹 정상 수 :", conflict_groups["Normal_Count"].sum())
print("충돌 그룹 불량 수 :", conflict_groups["Defect_Count"].sum())

print(
    "전체 불량 중 충돌 그룹 불량 비율 :",
    conflict_groups["Defect_Count"].sum() / (y_cn7 == 1).sum()
)

display(
    conflict_groups[
        [
            "Count",
            "Normal_Count",
            "Defect_Count"
        ]
    ]
)

충돌 X 그룹 수 : 11
충돌 그룹 전체 행 수 : 22
충돌 그룹 정상 수 : 11
충돌 그룹 불량 수 : 11
전체 불량 중 충돌 그룹 불량 비율 : 0.6470588235294118


,Count,Normal_Count,Defect_Count
398,2,1,1
428,2,1,1
486,2,1,1
495,2,1,1
529,2,1,1
532,2,1,1
536,2,1,1
550,2,1,1
576,2,1,1
590,2,1,1


불량 17건   
│   
├─ 6건: 정상과 명확하게 다른 X   
│      → 대부분 모델이 탐지   
│   
└─ 11건: 정상과 완전히 동일한 X 존재   
       → 모델이 X만 보고 구별 불가능   
       → 모델별 확률/순위 불안정   
       → ROS 실패   
       → Jitter 개선 없음   
       → 앙상블 개선 제한   

In [12]:
# 동일 X 정상 샘플이 존재하는 불량
conflict_defects = [
    47, 80, 95, 99, 101,
    103, 105, 107, 109, 111, 113
]

# 정상과 동일 X가 존재하지 않는 불량
non_conflict_defects = [
    115, 116, 117, 118, 119, 120
]

print("Conflict defects:", len(conflict_defects))
print("Non-conflict defects:", len(non_conflict_defects))

Conflict defects: 11
Non-conflict defects: 6


In [14]:
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)


# ============================================================
# 0. 데이터
# ============================================================

X_array = X_cn7.to_numpy(dtype=np.float32)
y_array = y_cn7.to_numpy()


# ============================================================
# 1. VAE 학습 함수
# ============================================================

def train_vae_model(
    X_train_normal,
    seed=42,
    epochs=100
):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = VAE(
        input_dim=X_train_normal.shape[1],
        latent_dim=8
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    dataset = TensorDataset(
        torch.tensor(
            X_train_normal,
            dtype=torch.float32
        )
    )

    loader = DataLoader(
        dataset,
        batch_size=64,
        shuffle=True
    )

    model.train()

    for epoch in range(epochs):

        for (batch,) in loader:

            optimizer.zero_grad()

            reconstructed, mu, logvar = model(
                batch
            )

            loss = vae_loss(
                reconstructed,
                batch,
                mu,
                logvar,
                beta=0.01
            )

            loss.backward()
            optimizer.step()

    return model


# ============================================================
# 2. VAE anomaly score
# ============================================================

def get_vae_score(model, X):

    model.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32
    )

    with torch.no_grad():

        # 평가에서는 random z 대신 mu 사용
        mu, logvar = model.encode(
            X_tensor
        )

        reconstructed = model.decoder(
            mu
        )

        score = torch.mean(
            (reconstructed - X_tensor) ** 2,
            dim=1
        )

    return score.numpy()


# ============================================================
# 3. Outer CV
# ============================================================

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

nested_vae_scores = np.zeros(
    len(y_array)
)

nested_vae_preds = np.zeros(
    len(y_array),
    dtype=int
)

fold_results = []


# ============================================================
# 4. Nested CV
# ============================================================

for outer_fold, (train_idx, val_idx) in enumerate(
    outer_cv.split(X_array, y_array),
    start=1
):

    print(f"Outer Fold {outer_fold}")

    X_train = X_array[train_idx]
    y_train = y_array[train_idx]

    X_val = X_array[val_idx]
    y_val = y_array[val_idx]


    # ========================================================
    # Inner CV → threshold 결정
    # ========================================================

    inner_cv = StratifiedKFold(
        n_splits=4,
        shuffle=True,
        random_state=42 + outer_fold
    )

    inner_scores = np.zeros(
        len(y_train)
    )


    for inner_fold, (
        inner_train_idx,
        inner_val_idx
    ) in enumerate(
        inner_cv.split(
            X_train,
            y_train
        ),
        start=1
    ):

        X_inner_train = X_train[
            inner_train_idx
        ]

        y_inner_train = y_train[
            inner_train_idx
        ]

        X_inner_val = X_train[
            inner_val_idx
        ]


        # 정상만 학습
        X_inner_normal = X_inner_train[
            y_inner_train == 0
        ]


        model = train_vae_model(
            X_inner_normal,
            seed=1000 + outer_fold * 10 + inner_fold
        )


        inner_scores[
            inner_val_idx
        ] = get_vae_score(
            model,
            X_inner_val
        )


    # ========================================================
    # Inner OOF에서 F1 최대 threshold
    # ========================================================

    thresholds = np.unique(
        inner_scores
    )

    best_threshold = None
    best_f1 = -1


    for threshold in thresholds:

        pred = (
            inner_scores >= threshold
        ).astype(int)

        score = f1_score(
            y_train,
            pred,
            zero_division=0
        )

        if score > best_f1:

            best_f1 = score
            best_threshold = threshold


    # ========================================================
    # Outer train 전체 정상으로 최종 학습
    # ========================================================

    X_train_normal = X_train[
        y_train == 0
    ]

    final_model = train_vae_model(
        X_train_normal,
        seed=2000 + outer_fold
    )


    val_scores = get_vae_score(
        final_model,
        X_val
    )

    val_preds = (
        val_scores >= best_threshold
    ).astype(int)


    # ★ 여기서 바로 저장
    nested_vae_scores[
        val_idx
    ] = val_scores

    nested_vae_preds[
        val_idx
    ] = val_preds


    fold_results.append({
        "Fold": outer_fold,
        "Threshold": best_threshold,
        "Inner_F1": best_f1
    })


# ============================================================
# 5. 전체 Nested CV 결과 확인
# ============================================================

print("\n[VAE Nested CV]")

print(
    "PR-AUC:",
    average_precision_score(
        y_array,
        nested_vae_scores
    )
)

print(
    "Precision:",
    precision_score(
        y_array,
        nested_vae_preds,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_array,
        nested_vae_preds,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_array,
        nested_vae_preds,
        zero_division=0
    )
)

display(
    pd.DataFrame(fold_results)
)

Outer Fold 1


NameError: name 'VAE' is not defined